**Author:** Stephanie Nord
# Unstructured SDS & Chemical Spec Sheet Parsing
**Objective:** Programmatically extract structured chemical entities (CAS Registry Numbers, GHS Hazard/Precautionary codes, and purity specs) from unstructured supplier documents using Python & Regular Expressions.

---
### **Pipeline Overview**
1. **Raw Ingestion:** Load messy, unstandardized chemical catalog text & SDS snippets.
2. **Regex Parsing:** Extract primary identifiers (`CAS`, `GHS H-Codes`, `GHS P-Codes`, `Purity %`).
3. **Structured Export:** Transform extracted entities into a clean, typed Pandas DataFrame.

In [1]:
import re
import pandas as pd

# Display settings so text and tables render cleanly in Jupyter
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print("✅ Setup complete! Libraries imported successfully.")

✅ Setup complete! Libraries imported successfully.


In [2]:
# Raw text samples mimicking unstructured supplier catalog feeds & SDS documents
raw_chemical_samples = [
    """
    PRODUCT SPECIFICATION SHEET
    Supplier: Sigma-Aldrich / Merck
    Product Name: Citric acid, anhydrous, 99.0%, ACS Reagent
    CAS Registry Number: 77-92-9
    GHS Hazard Statements: H319: Causes serious eye irritation. P264, P280
    Storage: Room Temp. Assay: >= 99.5%
    """,
    """
    SAFETY DATA SHEET (SDS)
    Item: Ethanol, Absolute, HPLC Grade, 99.9%
    CAS #: 64-17-5
    Hazards: H225 Highly flammable liquid and vapour. H319 Causes serious eye irritation.
    Precautionary: P210, P233, P240, P280
    Manufacturer: Thermo Fisher Scientific
    """,
    """
    RAW MATERIAL CATALOG FEED
    Vendor: VWR International
    Item Desc: Sodium Hydroxide Pellets, ACS Grade (CAS 1310-73-2)
    Purity Level: 98.0% min
    GHS Class: H314 Causes severe skin burns and eye damage. H290 May be corrosive to metals.
    Precautions: P260, P280, P305+P351+P338
    """,
    """
    TECHNICAL SPECIFICATION SHEET
    Supplier: Spectrum Chemical
    Product: Acetone, ReagentACS, 99.5%
    CAS: 67-64-1
    GHS Statements: H225, H319, H336
    Precautionary Statements: P210, P261, P305+P351+P338
    Form: Clear liquid
    """
]

print(f"✅ Ingested {len(raw_chemical_samples)} raw chemical text records.")

✅ Ingested 4 raw chemical text records.


## **Regex Field Extraction Engine**
Defining modular functions using Python's `re` library to extract primary identifiers:
* **CAS Registry Number:** Pattern `\b[1-9]\d{1,6}-\d{2}-\d\b`
* **GHS Hazard Codes (H-Codes):** Pattern `\bH\d{3}\b`
* **GHS Precautionary Codes (P-Codes):** Pattern `\bP\d{3}(?:\+P\d{3})*\b`
* **Purity Specs:** Pattern `\b\d{2}\.\d{1,2}%\b`

In [3]:
def extract_cas_number(text: str) -> str:
    """Extracts standard CAS Registry Number (e.g., 77-92-9 or 1310-73-2)."""
    pattern = r'\b[1-9]\d{1,6}-\d{2}-\d\b'
    match = re.search(pattern, text)
    return match.group(0) if match else "UNKNOWN"

def validate_cas_number(cas_number: str) -> bool:
    """Validates a CAS Registry Number using its check digit."""

    # Handle missing CAS numbers
    if cas_number == "UNKNOWN":
        return False
    
    # Remove hyphens
    digits = cas_number.replace("-", "")

    # Separate the check digit from the rest of the CAS number
    check_digit = int(digits[-1])
    body = digits[:-1]

    # Multiply digits from right to left by 1, 2, 3, ...
    total = sum(
        int(digit) * position
        for position, digit in enumerate(reversed(body), start = 1)
    )

    # A valid CAS number has a checksum matching the final digit
    return total % 10 == check_digit

def extract_ghs_h_codes(text: str) -> list:
    """Extracts all GHS Hazard Codes (e.g., H225, H319, H314)."""
    pattern = r'\bH\d{3}\b'
    return re.findall(pattern, text)

def extract_ghs_p_codes(text: str) -> list:
    """Extracts all GHS Precautionary Codes (e.g., P210, P280)."""
    pattern = r'\bP\d{3}(?:\+P\d{3})*\b'
    return re.findall(pattern, text)

def extract_purity(text: str) -> str:
    """Extracts purity percentage (e.g., 99.0%, 99.9%)."""
    pattern = r'\b\d{2}\.\d{1,2}%\b'
    match = re.search(pattern, text)
    return match.group(0) if match else "N/A"

def extract_grade(text: str) -> str:
    """Extracts chemical grade specification if present."""
    grade_map = {
    'ACS Reagent': 'ACS Reagent',
    'ACS Grade': 'ACS Grade',
    'HPLC Grade': 'HPLC Grade',
    'ReagentACS': 'ACS Reagent'
}
    for source_grade, standardized_grade in grade_map.items():
        if re.search(r'\b' + re.escape(source_grade) + r'\b', text, re.IGNORECASE):
            return standardized_grade
    return "Standard / Unspecified"

print("✅ Regex parsing functions defined and loaded!")

✅ Regex parsing functions defined and loaded!


In [4]:
parsed_records = []

for sample in raw_chemical_samples:
    # Clean up newline artifacts for uniform processing
    clean_text = sample.strip().replace('\n', ' ')

    # Extract CAS number once, then validate it
    cas_number = extract_cas_number(clean_text)

    record = {
        "cas_number": cas_number,
        "cas_valid": validate_cas_number(cas_number),
        "purity_spec": extract_purity(clean_text),
        "grade_spec": extract_grade(clean_text),
        "hazard_h_codes": ", ".join(extract_ghs_h_codes(clean_text)),
        "precaution_p_codes": ", ".join(extract_ghs_p_codes(clean_text)),
        "raw_text": clean_text,
        "raw_text_snippet": clean_text[:80] + "..."
    }

    parsed_records.append(record)

# Convert parsed dicts into a structured Pandas DataFrame
df_parsed = pd.DataFrame(parsed_records)

# Display the final structured output table
df_parsed[
    [
        "cas_number",
        "cas_valid",
        "purity_spec",
        "grade_spec",
        "hazard_h_codes",
        "precaution_p_codes"
    ]
]

,cas_number,cas_valid,purity_spec,grade_spec,hazard_h_codes,precaution_p_codes
0,77-92-9,True,N/A,ACS Reagent,H319,"P264, P280"
1,64-17-5,True,N/A,HPLC Grade,"H225, H319","P210, P233, P240, P280"
2,1310-73-2,True,N/A,ACS Grade,"H314, H290","P260, P280, P305+P351+P338"
3,67-64-1,True,N/A,ACS Reagent,"H225, H319, H336","P210, P261, P305+P351+P338"


### **Iterative Refinement: Handling Edge Cases in Chemical Purity Extraction**

Data pipelines handling unstructured vendor text frequently encounter formatting inconsistencies across different chemical suppliers.

#### **1. The Parsing Failure in v1**
In our initial extraction pass, `purity_spec` returned `N/A` for all records due to strict regex constraints:
* **Word Boundary Mismatches:** The pattern `\b\d{2}\.\d{1,2}%\b` relied on `%\b`. Because `%` is a non-word character, `%\b` fails when adjacent to spaces, punctuation, or string ends.
* **Rigid Digit Requirements:** Expecting exactly two digits before the decimal failed on integers or non-standard formatting.

#### **2. Domain-Specific Purity Representations**
In chemical manufacturing and procurement, purity specifications are written in several standard industry formats:
* **Assay Minimums / Relational Operators:** `>= 99.5%` or `≥ 98.0% min` (common in industrial and bulk technical grades).
* **High-Purity Decimals:** `99.99%` or `99.999%` (critical for trace metal analysis and HPLC/semiconductor applications).
* **Integer Percentages:** `99%` (often used in raw catalog listings without floating-point decimals).

#### **3. Upgrading to Regex Engine v2**
The updated function uses `(?:>=|>|≥)?\s*\b\d{2,3}(?:\.\d{1,3})?\s*%`:
* **`(?:>=|>|≥)?`**: An optional non-capturing group for relational operators.
* **`\d{2,3}`**: Flexibly handles 2 to 3 integer digits (up to 100%).
* **`(?:\.\d{1,3})?`**: An optional non-capturing group for up to 3 decimal places.
* **`\s*%`**: Captures the percentage sign even if there is whitespace.

In [5]:
def extract_purity_v2(text: str) -> str:
    """Refined regex to capture percentages with optional leading operators (>=, >) 

    and flexible decimal lengths (e.g., 99%, 99.5%, >= 99.0%).
    """
    # Captures optional >= or >, followed by 2 to 3 digits, optional decimal, and %
    pattern = r'(?:>=|>|≥)?\s*\b\d{2,3}(?:\.\d{1,3})?\s*%'
    match = re.search(pattern, text)
    return match.group(0).strip() if match else "N/A"

# Test the updated function on our raw samples
for i, sample in enumerate(raw_chemical_samples):
    print(f"Row {i} Fixed Purity: {extract_purity_v2(sample)}")

Row 0 Fixed Purity: 99.0%
Row 1 Fixed Purity: 99.9%
Row 2 Fixed Purity: 98.0%
Row 3 Fixed Purity: 99.5%


In [6]:
# 1. Update the purity column across all parsed records using our v2 function
for record in parsed_records:
    record["purity_spec"] = extract_purity_v2(record["raw_text"])

# Re-create the DataFrame with updated purity specs
df_parsed_v2 = pd.DataFrame(parsed_records)

# Select and order final production columns
final_cols = ['cas_number', 'cas_valid', 'purity_spec', 'grade_spec', 'hazard_h_codes', 'precaution_p_codes']
df_final = df_parsed_v2[final_cols]

# Display the updated clean table
print("=== FINAL PARSED CHEMICAL SPECIFICATIONS ===")
display(df_final)

# 2. Export clean structured data to CSV
csv_filename = "parsed_sds_specifications.csv"
df_final.to_csv(csv_filename, index=False)

print(f"\n✅ Pipeline complete! Clean data successfully exported to '{csv_filename}'.")

=== FINAL PARSED CHEMICAL SPECIFICATIONS ===


,cas_number,cas_valid,purity_spec,grade_spec,hazard_h_codes,precaution_p_codes
0,77-92-9,True,99.0%,ACS Reagent,H319,"P264, P280"
1,64-17-5,True,99.9%,HPLC Grade,"H225, H319","P210, P233, P240, P280"
2,1310-73-2,True,98.0%,ACS Grade,"H314, H290","P260, P280, P305+P351+P338"
3,67-64-1,True,99.5%,ACS Reagent,"H225, H319, H336","P210, P261, P305+P351+P338"



✅ Pipeline complete! Clean data successfully exported to 'parsed_sds_specifications.csv'.
